# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:

# Milestone 1: Is a Famous Movie Necessarily a Good Movie?

## Exploring the Relationship Between Audience Attention and Audience Approval

Movies can be evaluated in different ways. Some measures reflect how much attention a movie receives, while others reflect how positively audiences rate it. What is not yet clear is how closely these two dimensions align.

This project investigates the relationship between **audience attention** and **audience approval** across movies.

Audience attention will refer to the level of audience activity or visibility associated with a movie, while audience approval will refer to how positively viewers rate that movie. The specific variables used to represent these concepts will be selected after the available data and their distributions are examined.

### Main Research Question

**How are audience attention and audience approval related across movies, and which movie characteristics are associated with different attention–approval patterns?**

The intended user is a streaming platform content strategy or catalog management team. Understanding whether attention and approval move together, differ, or form distinct patterns could help such a team better understand its catalog and audience response.

The analysis will explore several supporting questions:

1. How strongly are audience attention and audience approval associated across movies?
2. Do movies with greater audience attention also tend to receive higher audience ratings?
3. Are there groups of movies that show different combinations of attention and approval?
4. Are movie characteristics such as genre, language, runtime, release period, director, cast, or keywords associated with these patterns?
5. How does the number of ratings behind a movie affect the reliability of its average audience rating?

One possible pattern of interest is whether some movies receive strong audience approval despite relatively limited attention. If such a pattern exists in the data, these movies could later be examined as potential **hidden gems**. However, this classification will not be assumed in advance.

 Because the available data are observational, any relationship found between movie characteristics, attention, and audience approval will not be interpreted as evidence of causatio and will be investigated.

At this stage, terms such as **high attention**, **high approval**, and **hidden gem** will not be assigned fixed thresholds. Their definitions, if used later, will be based on the observed distributions and quality of the data.

The planned final unit of analysis is **one movie per row**. Since the raw datasets contain information at different levels of granularity, the first step is to understand what each source represents, evaluate its quality, and determine how the sources can later be combined into a reliable movie-level analytical dataset.

## The Data Behind the Story

To answer whether a famous movie is necessarily a highly rated movie, the project uses five related datasets. Each file contributes a different part of the analysis, and understanding what one row represents in each table is important before the files are combined.

### 1. `movies_metadata.csv`

This is the main movie-level dataset.

***One row represents one movie.***

The table contains movie characteristics such as title, language, release information, runtime, budget, revenue, popularity, vote average, genre-related fields, and production information.

Important identifiers include:

- `movieId`
- `tmdbId`

This table will serve as the main foundation for the final movie-level analytical dataset.

### 2. `ratings.csv`

This table contains individual audience ratings.

***One row represents one user rating one movie.***

Important fields include:

- `userId`
- `movieId`
- `rating`
- `rating_datetime`

A single movie could possibly appear many times in this table because multiple users can rate the same movie. This means the ratings table has a different level of granularity from the movie metadata.

Before ratings can be combined with the movie-level data, they will later need to be summarized by `movieId`.

### 3. `links.csv`

This table provides identifier mappings between movie identification systems.

***One row represents one movie identifier mapping.***

The table can be used to connect MovieLens-style movie identifiers with external identifiers such as TMDB IDs.

Because the movie metadata already contains the main identifiers needed for this project, this table will initially be used mainly for validation and identifier consistency checks.

### 4. `credits_curated.csv`

This table contains curated information about the people involved in each movie.

***One row represents one movie's curated credit information.***

It includes information such as the director and major cast members and connects to the movie metadata through `tmdbId`.

This information may later help evaluate whether director or cast characteristics are associated with different attention–approval patterns.

### 5. `keywords_curated.csv`

This table contains descriptive keywords associated with each movie.

***One row represents one movie's curated keyword information.***

The table connects to the movie metadata using `tmdbId`.

Keywords may later help describe movie themes or content characteristics that are not fully captured by broader categories such as genre.

## Granularity 

The five datasets do not all describe the observations at the same level.

The movie metadata, credits, and keywords are used to represent movies at the movie level, while the ratings table contains repeated observations for each movie because every user rating is stored separately.

Directly joining raw ratings to movie metadata would create multiple rows for the same movie.

The planned final analytical dataset will therefore use **one row per movie**. Before that dataset is created, the ratings information will need to be aggregated to the movie level.

In [6]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
primary_genre,0.003854
genres_list,0.003854
status,0.000220
movieId,0.000000
vote_average,0.000000
adult,0.000000
production_countries,0.000000
production_companies,0.000000
genres,0.000000
vote_count,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


...

,dtype
tmdbId,int64
director,str
top_cast,str


,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Missing Values

A dataset can contain the correct variables and still be difficult to use if important information is frequently missing.

The next step examines missing values in each raw table. Both the number and percentage of missing observations are considered because the same missing count can have very different importance depending on the size of the dataset.

Missing values will be identified and documented and not automatically removed. Decisions about dropping, retaining, or transforming variables will be made after considering how much information is missing and how important each variable is to the project.

In [7]:
# ============================================================
# STEP 4: MISSING VALUES PROFILE
# ============================================================

for name, df in tables.items():

    missing_profile = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    missing_profile = (
        missing_profile
        .sort_values("missing_percent", ascending=False)
    )

    print("\n" + "=" * 70)
    print(f"{name.upper()} - MISSING VALUES")
    print("=" * 70)

    display(missing_profile)


MOVIES - MISSING VALUES


,missing_count,missing_percent
primary_genre,35,0.39
genres_list,35,0.39
status,2,0.02
movieId,0,0.00
vote_average,0,0.00
adult,0,0.00
production_countries,0,0.00
production_companies,0,0.00
genres,0,0.00
vote_count,0,0.00



RATINGS - MISSING VALUES


,missing_count,missing_percent
userId,0,0.0
movieId,0,0.0
rating,0,0.0
timestamp,0,0.0
rating_datetime,0,0.0



LINKS - MISSING VALUES


,missing_count,missing_percent
tmdbId,13,0.14
movieId,0,0.00
imdbId,0,0.00



CREDITS - MISSING VALUES


,missing_count,missing_percent
top_cast,90,0.99
director,24,0.26
tmdbId,0,0.00



KEYWORDS - MISSING VALUES


,missing_count,missing_percent
keywords_list,761,8.38
tmdbId,0,0.00


In [8]:
missing_summary = []

for name, df in tables.items():

    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())

    missing_summary.append({
        "table": name,
        "total_cells": total_cells,
        "missing_cells": missing_cells,
        "missing_percent": round(
            (missing_cells / total_cells) * 100,
            2
        )
    })

missing_summary = pd.DataFrame(missing_summary)

display(missing_summary)

,table,total_cells,missing_cells,missing_percent
0,movies,190722,72,0.04
1,ratings,499050,0,0.00
2,links,27375,13,0.05
3,credits,27246,114,0.42
4,keywords,18164,761,4.19


## Duplicate Keys and Identifier Integrity

The next step checks whether the identifiers in each table behave as expected.

A table can contain no fully duplicated records and still have repeated values in a field that is expected to uniquely identify one observation.

For the movie-level tables, repeated movie identifiers could create problems during later joins. The ratings table is different because repeated `movieId` values are expected; many users can rate the same movie. For that reason, the combined `userId` and `movieId` fields are used to check rating-level duplication.

This audit will also check whether any expected key values are missing.

In [9]:
# ============================================================
# STEP 5: DUPLICATE KEYS AND IDENTIFIER INTEGRITY
# ============================================================

key_audit = pd.DataFrame([
    {
        "table": "movies",
        "expected_key": "movieId",
        "duplicate_keys": movies["movieId"].duplicated().sum(),
        "missing_key_values": movies["movieId"].isna().sum(),
        "unique_keys": movies["movieId"].nunique()
    },
    {
        "table": "movies",
        "expected_key": "tmdbId",
        "duplicate_keys": movies["tmdbId"].duplicated().sum(),
        "missing_key_values": movies["tmdbId"].isna().sum(),
        "unique_keys": movies["tmdbId"].nunique()
    },
    {
        "table": "ratings",
        "expected_key": "userId + movieId",
        "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(),
        "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum(),
        "unique_keys": ratings[["userId", "movieId"]].drop_duplicates().shape[0]
    },
    {
        "table": "links",
        "expected_key": "movieId",
        "duplicate_keys": links["movieId"].duplicated().sum(),
        "missing_key_values": links["movieId"].isna().sum(),
        "unique_keys": links["movieId"].nunique()
    },
    {
        "table": "credits",
        "expected_key": "tmdbId",
        "duplicate_keys": credits["tmdbId"].duplicated().sum(),
        "missing_key_values": credits["tmdbId"].isna().sum(),
        "unique_keys": credits["tmdbId"].nunique()
    },
    {
        "table": "keywords",
        "expected_key": "tmdbId",
        "duplicate_keys": keywords["tmdbId"].duplicated().sum(),
        "missing_key_values": keywords["tmdbId"].isna().sum(),
        "unique_keys": keywords["tmdbId"].nunique()
    }
])

display(key_audit)

,table,expected_key,duplicate_keys,missing_key_values,unique_keys
0,movies,movieId,0,0,9082
1,movies,tmdbId,0,0,9082
2,ratings,userId + movieId,0,0,99810
3,links,movieId,0,0,9125
4,credits,tmdbId,0,0,9082
5,keywords,tmdbId,0,0,9082


In [10]:
# Show duplicated key records only if they exist

if movies["movieId"].duplicated().any():
    display(
        movies[movies["movieId"].duplicated(keep=False)]
        .sort_values("movieId")
    )

if movies["tmdbId"].duplicated().any():
    display(
        movies[movies["tmdbId"].duplicated(keep=False)]
        .sort_values("tmdbId")
    )

if ratings[["userId", "movieId"]].duplicated().any():
    display(
        ratings[
            ratings[["userId", "movieId"]]
            .duplicated(keep=False)
        ]
        .sort_values(["userId", "movieId"])
        .head(20)
    )

### Key Audit Interpretation

The key audit did not identify any duplicated or missing values in the expected identifiers.

The `movies`, `credits`, and `keywords` tables each contain 9,082 unique movie identifiers, while the `links` table contains 9,125 unique `movieId` values. The difference in the number of records between `links` and the other movie-level tables will be examined later when table coverage is evaluated.

The ratings table contains 99,810 unique `userId`–`movieId` combinations, with no duplicated combinations or missing key values. This suggests that each row represents a distinct user–movie rating event in the available data.

These results are encouraging for later integration because the movie-level tables currently show unique keys, which supports one-to-one joins where appropriate. The difference in coverage across tables will still need to be validated before the final analytical dataset is created.

## Looking for Suspicious or Confusing Values

Structural checks can show whether a dataset is organized correctly, but they do not tell us whether the values themselves are reasonable.

The next step examines important numeric and date-related variables for values that may require further investigation. The goal is not to automatically remove unusual observations. Instead, the analysis will distinguish between legitimate extreme values, missing-like values, and observations that may indicate a data-quality problem.

Particular attention will be given to audience ratings, runtime, budget, revenue, popularity, vote-related fields, and release information.

In [11]:
# ============================================================
# STEP 6: SUSPICIOUS AND CONFUSING VALUES
# ============================================================

# Check audience rating values
print("=" * 70)
print("RATINGS DISTRIBUTION")
print("=" * 70)

display(ratings["rating"].describe())

print("\nUnique rating values:")
print(sorted(ratings["rating"].dropna().unique()))

RATINGS DISTRIBUTION


count    99810.000000
mean         3.542972
std          1.057939
min          0.500000
25%          3.000000
50%          4.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64


Unique rating values:
[np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]


In [12]:
# Check important numeric movie variables

numeric_movie_cols = [
    "budget",
    "revenue",
    "runtime",
    "popularity",
    "vote_average",
    "vote_count"
]

available_numeric_cols = [
    col for col in numeric_movie_cols
    if col in movies.columns
]

numeric_checks = []

for col in available_numeric_cols:
    values = pd.to_numeric(movies[col], errors="coerce")

    numeric_checks.append({
        "variable": col,
        "missing_after_conversion": int(values.isna().sum()),
        "zero_values": int((values == 0).sum()),
        "negative_values": int((values < 0).sum()),
        "minimum": values.min(),
        "median": values.median(),
        "maximum": values.max()
    })

numeric_checks = pd.DataFrame(numeric_checks)

display(numeric_checks)

,variable,missing_after_conversion,zero_values,negative_values,minimum,median,maximum
0,budget,0,4443,0,0.000000,100000.000000,3.800000e+08
1,revenue,0,4322,0,0.000000,271608.000000,2.787965e+09
2,runtime,0,20,0,0.000000,102.000000,1.140000e+03
3,popularity,0,0,0,0.000004,6.418871,5.474883e+02
4,vote_average,0,48,0,0.000000,6.500000,1.000000e+01
5,vote_count,0,44,0,0.000000,96.000000,1.407500e+04


In [13]:
# Check release date values

if "release_date" in movies.columns:

    release_dates = pd.to_datetime(
        movies["release_date"],
        errors="coerce"
    )

    release_date_check = pd.Series({
        "missing_or_invalid_dates": release_dates.isna().sum(),
        "earliest_release_date": release_dates.min(),
        "latest_release_date": release_dates.max()
    })

    display(release_date_check)

missing_or_invalid_dates                      0
earliest_release_date       1902-09-01 00:00:00
latest_release_date         2016-09-23 00:00:00
dtype: object

### Suspicious Value Interpretation

The initial value checks identified several observations that should be investigated during cleaning, but none should be removed automatically at this stage.

The individual audience ratings range from **0.5 to 5.0**, with a mean of approximately **3.54** and a median of **4.0**. The observed minimum and maximum are consistent with the rating scale represented in the dataset.

Budget and revenue contain a large number of zero values. Of the 9,082 movies, **4,443 have a recorded budget of zero** and **4,322 have recorded revenue of zero**. Since zero may represent unavailable financial information instead of a true zero-dollar budget or revenue, these values will require special treatment before the variables are used in later analysis.

Runtime also contains **20 zero values**, which are questionable for a movie-level dataset and should be investigated. The maximum runtime is **1,140 minutes**, which is unusually large and should be examined to determine whether it represents a legitimate observation or an unusual record.

`vote_average` contains **48 zero values**, and `vote_count` contains **44 zero values**. These observations may represent movies with little or no voting activity and should be interpreted carefully if these variables are used as measures of audience attention or approval.

Popularity does not contain zero values in this initial check, although its values range from very small values to approximately **547.49**, suggesting a strongly uneven distribution that should be examined before deciding how popularity will be used.

The release-date field produced no missing or invalid dates during conversion. The movies range from **September 1, 1902 to September 23, 2016**, indicating that the dataset covers more than a century of film releases. This wide time span may become important later because older movies have had different opportunities to accumulate ratings and audience attention than newer releases.

Overall, these checks suggest that the main concerns are not negative values or failed numeric conversions, but the interpretation of zeros, unusually large observations, and differences in exposure across movies.

## How Much Evidence Is Behind a Movie's Rating?

An average rating does not carry the same amount of evidence for every movie.

A movie rated by only a few users may have a very high average rating, while another movie's average may be based on hundreds or thousands of audience responses. Before using average rating as a measure of audience approval, it is therefore important to understand how many ratings each movie has received.

This step summarizes the individual rating records at the movie level for profiling purposes. It does not yet create the final analytical base table. The goal is to examine how rating volume varies across movies and determine whether a minimum amount of rating evidence may eventually be needed before interpreting a movie's average rating.

In [14]:
# ============================================================
# STEP 7: MOVIE-LEVEL RATING PROFILE
# ============================================================

movie_rating_profile = (
    ratings
    .groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        rating_mean=("rating", "mean"),
        rating_std=("rating", "std"),
        unique_users=("userId", "nunique")
    )
    .reset_index()
)

print("Number of movies with at least one rating:")
print(f"{len(movie_rating_profile):,}")

print("\nRating count per movie:")
display(movie_rating_profile["rating_count"].describe())

print("\nAverage rating across movies:")
display(movie_rating_profile["rating_mean"].describe())

Number of movies with at least one rating:
9,025

Rating count per movie:


count    9025.000000
mean       11.059280
std        24.095433
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       341.000000
Name: rating_count, dtype: float64


Average rating across movies:


count    9025.000000
mean        3.290203
std         0.881612
min         0.500000
25%         2.833333
50%         3.500000
75%         3.961538
max         5.000000
Name: rating_mean, dtype: float64

In [15]:
# How much rating evidence does each movie have?

rating_evidence = pd.Series({
    "1+ ratings": (movie_rating_profile["rating_count"] >= 1).sum(),
    "5+ ratings": (movie_rating_profile["rating_count"] >= 5).sum(),
    "10+ ratings": (movie_rating_profile["rating_count"] >= 10).sum(),
    "20+ ratings": (movie_rating_profile["rating_count"] >= 20).sum(),
    "50+ ratings": (movie_rating_profile["rating_count"] >= 50).sum(),
    "100+ ratings": (movie_rating_profile["rating_count"] >= 100).sum(),
    "500+ ratings": (movie_rating_profile["rating_count"] >= 500).sum()
})

display(
    rating_evidence
    .to_frame("number_of_movies")
)

,number_of_movies
1+ ratings,9025
5+ ratings,3485
10+ ratings,2241
20+ ratings,1300
50+ ratings,453
100+ ratings,151
500+ ratings,0


In [16]:
# Movies with the fewest and most ratings

print("Movies with the fewest ratings:")
display(
    movie_rating_profile
    .sort_values("rating_count")
    .head(10)
)

print("\nMovies with the most ratings:")
display(
    movie_rating_profile
    .sort_values("rating_count", ascending=False)
    .head(10)
)

Movies with the fewest ratings:


,movieId,rating_count,rating_mean,rating_std,unique_users
9024,163949,1,5.0,NaN,1
5770,26422,1,5.0,NaN,1
2403,2998,1,4.0,NaN,1
5769,26414,1,3.5,NaN,1
2405,3001,1,2.0,NaN,1
5768,26413,1,4.0,NaN,1
2407,3003,1,1.0,NaN,1
5767,26409,1,1.0,NaN,1
5766,26404,1,2.5,NaN,1
5764,26400,1,4.5,NaN,1



Movies with the most ratings:


,movieId,rating_count,rating_mean,rating_std,unique_users
321,356,341,4.054252,0.871521,341
266,296,324,4.256173,0.866897,324
284,318,311,4.487138,0.702412,311
525,593,304,4.138158,0.874459,304
232,260,291,4.221649,0.908682,291
427,480,274,3.706204,0.917073,274
2058,2571,259,4.183398,0.901202,259
0,1,247,3.872470,0.958981,247
472,527,244,4.303279,0.882051,244
522,589,237,4.006329,0.879957,237


## How Reliable Is a Movie's Average Rating?

Average audience rating can be misleading when only a small number of users have rated a movie.

For example, a movie with an average rating of 4.8 based on five users does not provide the same amount of evidence as a movie with an average rating of 4.5 based on several thousand users.

Before audience approval is used as an analytical measure, the distribution of ratings per movie should therefore be examined. This step summarizes the rating records at the movie level to understand how much rating evidence exists behind each movie's average score.

At this stage, no minimum rating threshold will be selected. The purpose is first to examine the distribution and then determine whether a threshold is justified by the data.

In [17]:
# ============================================================
# STEP 7: MOVIE-LEVEL RATING RELIABILITY PROFILE
# ============================================================

movie_rating_profile = (
    ratings
    .groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        rating_mean=("rating", "mean"),
        rating_std=("rating", "std"),
        unique_users=("userId", "nunique")
    )
    .reset_index()
)

print("Number of movies with at least one rating:")
print(f"{len(movie_rating_profile):,}")

print("\nNumber of ratings per movie:")
display(movie_rating_profile["rating_count"].describe())

print("\nAverage rating across movies:")
display(movie_rating_profile["rating_mean"].describe())

Number of movies with at least one rating:
9,025

Number of ratings per movie:


count    9025.000000
mean       11.059280
std        24.095433
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       341.000000
Name: rating_count, dtype: float64


Average rating across movies:


count    9025.000000
mean        3.290203
std         0.881612
min         0.500000
25%         2.833333
50%         3.500000
75%         3.961538
max         5.000000
Name: rating_mean, dtype: float64

In [18]:
# Number of movies meeting different rating-count levels

rating_evidence = pd.DataFrame({
    "minimum_rating_count": [1, 5, 10, 20, 50, 100, 500],
    "number_of_movies": [
        (movie_rating_profile["rating_count"] >= 1).sum(),
        (movie_rating_profile["rating_count"] >= 5).sum(),
        (movie_rating_profile["rating_count"] >= 10).sum(),
        (movie_rating_profile["rating_count"] >= 20).sum(),
        (movie_rating_profile["rating_count"] >= 50).sum(),
        (movie_rating_profile["rating_count"] >= 100).sum(),
        (movie_rating_profile["rating_count"] >= 500).sum()
    ]
})

rating_evidence["percent_of_rated_movies"] = (
    rating_evidence["number_of_movies"]
    / len(movie_rating_profile)
    * 100
).round(2)

display(rating_evidence)

,minimum_rating_count,number_of_movies,percent_of_rated_movies
0,1,9025,100.00
1,5,3485,38.61
2,10,2241,24.83
3,20,1300,14.40
4,50,453,5.02
5,100,151,1.67
6,500,0,0.00


In [19]:
# Inspect movies at both ends of the rating-count distribution

print("Movies with the fewest ratings:")
display(
    movie_rating_profile
    .sort_values("rating_count")
    .head(10)
)

print("\nMovies with the most ratings:")
display(
    movie_rating_profile
    .sort_values("rating_count", ascending=False)
    .head(10)
)

Movies with the fewest ratings:


,movieId,rating_count,rating_mean,rating_std,unique_users
9024,163949,1,5.0,NaN,1
5770,26422,1,5.0,NaN,1
2403,2998,1,4.0,NaN,1
5769,26414,1,3.5,NaN,1
2405,3001,1,2.0,NaN,1
5768,26413,1,4.0,NaN,1
2407,3003,1,1.0,NaN,1
5767,26409,1,1.0,NaN,1
5766,26404,1,2.5,NaN,1
5764,26400,1,4.5,NaN,1



Movies with the most ratings:


,movieId,rating_count,rating_mean,rating_std,unique_users
321,356,341,4.054252,0.871521,341
266,296,324,4.256173,0.866897,324
284,318,311,4.487138,0.702412,311
525,593,304,4.138158,0.874459,304
232,260,291,4.221649,0.908682,291
427,480,274,3.706204,0.917073,274
2058,2571,259,4.183398,0.901202,259
0,1,247,3.872470,0.958981,247
472,527,244,4.303279,0.882051,244
522,589,237,4.006329,0.879957,237


## What Does “Famous” Mean in This Dataset?

The project title raises an important measurement question: how should a movie's fame or audience attention be represented using the available data?

Several variables may capture related but different aspects of attention. These include the number of user ratings a movie receives, its popularity measure, and its vote count. At the same time, average user rating may provide a possible measure of audience approval.

These variables should not automatically be treated as equivalent. Before selecting the measures used in later analysis, their distributions and relationships need to be examined.

For this initial profile:

- `rating_count` represents how many rating events are available for a movie.
- `rating_mean` represents the average MovieLens user rating.
- `popularity` is an available movie-level popularity measure.
- `vote_count` represents the number of votes recorded in the movie metadata.
- `vote_average` provides another audience-rating measure available in the metadata.



In [20]:
# ============================================================
# STEP 8: CANDIDATE ATTENTION AND APPROVAL MEASURES
# ============================================================

# Create a temporary profiling table only for Milestone 1 exploration
attention_approval_profile = (
    movies[
        [
            "movieId",
            "title",
            "popularity",
            "vote_count",
            "vote_average"
        ]
    ]
    .merge(
        movie_rating_profile[
            [
                "movieId",
                "rating_count",
                "rating_mean"
            ]
        ],
        on="movieId",
        how="left"
    )
)

candidate_measures = [
    "popularity",
    "vote_count",
    "vote_average",
    "rating_count",
    "rating_mean"
]

display(
    attention_approval_profile[
        candidate_measures
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
popularity,9082.0,7.268905,9.372314,0.000004,2.954521,6.418871,9.861742,547.488298
vote_count,9082.0,439.050870,996.156492,0.000000,29.000000,96.000000,365.750000,14075.000000
vote_average,9082.0,6.362178,1.038748,0.000000,5.800000,6.500000,7.000000,10.000000
rating_count,9025.0,11.059280,24.095433,1.000000,1.000000,3.000000,9.000000,341.000000
rating_mean,9025.0,3.290203,0.881612,0.500000,2.833333,3.500000,3.961538,5.000000


### Initial Attention–Approval Interpretation

The candidate measures do not appear to represent the same concept.

The three potential attention measures show positive relationships with one another, although their relationships vary in strength. `popularity` and `vote_count` have a correlation of 0.539, while `vote_count` and MovieLens `rating_count` have a correlation of 0.511. The relationship between `popularity` and `rating_count` is weaker at 0.288.

The initial relationships between these attention measures and MovieLens average rating are much weaker. `popularity` has a correlation of approximately -0.004 with `rating_mean`, `vote_count` has a correlation of 0.042, and `rating_count` has a correlation of 0.132.

These results do not establish that attention and approval are unrelated. They provide early evidence that the available measures may capture different dimensions of movie performance and support examining the relationship more carefully in later milestones.

The metadata `vote_average` and MovieLens `rating_mean` show a stronger positive correlation of 0.433. Because these are two separate rating measures, their definitions and sources should be considered before deciding how they will be used in the final analysis.

For now, no single variable will automatically be labeled as the definitive measure of “fame.”

In [21]:
# Correlations among candidate attention and approval measures

candidate_correlations = (
    attention_approval_profile[
        candidate_measures
    ]
    .corr()
    .round(3)
)

display(candidate_correlations)

,popularity,vote_count,vote_average,rating_count,rating_mean
popularity,1.000,0.539,0.122,0.288,-0.004
vote_count,0.539,1.000,0.186,0.511,0.042
vote_average,0.122,0.186,1.000,0.227,0.433
rating_count,0.288,0.511,0.227,1.000,0.132
rating_mean,-0.004,0.042,0.433,0.132,1.000


In [22]:
# Inspect movies with the highest values for different candidate measures

print("Highest popularity:")
display(
    attention_approval_profile
    .sort_values("popularity", ascending=False)
    .head(10)
)

print("\nHighest vote count:")
display(
    attention_approval_profile
    .sort_values("vote_count", ascending=False)
    .head(10)
)

print("\nHighest MovieLens rating count:")
display(
    attention_approval_profile
    .sort_values("rating_count", ascending=False)
    .head(10)
)

print("\nHighest MovieLens average rating:")
display(
    attention_approval_profile
    .sort_values("rating_mean", ascending=False)
    .head(10)
)

Highest popularity:


,movieId,title,popularity,vote_count,vote_average,rating_count,rating_mean
8889,135887,Minions,547.488298,4729.0,6.4,5.0,2.900000
8664,115617,Big Hero 6,213.849907,6289.0,7.8,21.0,4.095238
8752,122904,Deadpool,187.860492,11444.0,7.4,18.0,3.388889
7390,72998,Avatar,185.070892,12114.0,7.2,67.0,3.641791
8654,115149,John Wick,183.870374,5499.0,7.0,10.0,3.450000
8586,112556,Gone Girl,154.801009,6023.0,7.9,25.0,3.720000
8683,116823,The Hunger Games: Mockingjay - Part 1,147.098006,5767.0,6.6,10.0,2.900000
8753,122920,Captain America: Civil War,145.882135,7462.0,7.1,7.0,3.571429
266,296,Pulp Fiction,140.950236,8670.0,8.3,324.0,4.256173
6897,58559,The Dark Knight,123.167259,12269.0,8.3,121.0,4.235537



Highest vote count:


,movieId,title,popularity,vote_count,vote_average,rating_count,rating_mean
7550,79132,Inception,29.108149,14075.0,8.1,111.0,4.049550
6897,58559,The Dark Knight,123.167259,12269.0,8.3,121.0,4.235537
7390,72998,Avatar,185.070892,12114.0,7.2,67.0,3.641791
7864,89745,The Avengers,89.887648,12000.0,7.4,46.0,4.010870
8752,122904,Deadpool,187.860492,11444.0,7.4,18.0,3.388889
8501,109487,Interstellar,32.213481,11187.0,8.1,43.0,4.151163
8198,99114,Django Unchained,19.785025,10297.0,7.8,44.0,3.977273
8600,112852,Guardians of the Galaxy,53.291601,10014.0,7.9,37.0,3.986486
2369,2959,Fight Club,63.869599,9678.0,8.3,202.0,4.178218
7924,91500,The Hunger Games,20.031667,9634.0,6.9,38.0,3.381579



Highest MovieLens rating count:


,movieId,title,popularity,vote_count,vote_average,rating_count,rating_mean
321,356,Forrest Gump,48.307194,8147.0,8.2,341.0,4.054252
266,296,Pulp Fiction,140.950236,8670.0,8.3,324.0,4.256173
284,318,The Shawshank Redemption,51.645403,8358.0,8.5,311.0,4.487138
525,593,The Silence of the Lambs,4.307222,4549.0,8.1,304.0,4.138158
232,260,Star Wars,42.149697,6778.0,8.1,291.0,4.221649
427,480,Jurassic Park,8.863776,4956.0,7.6,274.0,3.706204
2058,2571,The Matrix,33.366332,9079.0,7.9,259.0,4.183398
0,1,Toy Story,21.946943,5415.0,7.7,247.0,3.872470
472,527,Schindler's List,41.725123,4436.0,8.3,244.0,4.303279
522,589,Terminator 2: Judgment Day,22.661695,4274.0,7.7,237.0,4.006329



Highest MovieLens average rating:


,movieId,title,popularity,vote_count,vote_average,rating_count,rating_mean
9081,163949,The Beatles: Eight Days a Week - The Touring Y...,7.078301,92.0,7.6,1.0,5.0
8271,101850,The Staircase,0.779002,22.0,7.9,1.0,5.0
5732,26151,Au Hasard Balthazar,4.530601,74.0,7.8,1.0,5.0
5731,26150,Andrei Rublev,6.489852,139.0,8.0,1.0,5.0
8219,99764,It's Such a Beautiful Day,3.974088,94.0,7.8,2.0,5.0
5722,26094,L'eclisse,5.656700,68.0,7.7,1.0,5.0
1409,1819,Storefront Hitchcock,0.036965,1.0,7.0,1.0,5.0
8247,100553,Frozen Planet,3.801630,46.0,8.3,1.0,5.0
5674,25852,Gaslight,1.102931,20.0,7.1,1.0,5.0
1433,1859,Taste of Cherry,7.894941,78.0,7.1,2.0,5.0


### Rating Reliability Interpretation

The rating-count distribution shows that audience approval is supported by very different amounts of evidence across movies. Although 9,025 movies have at least one rating, the median movie has only three ratings, and 75% of rated movies have nine or fewer.

Only 1,300 movies, or 14.4% of rated movies, have at least 20 ratings. This means that a high average rating alone may not provide a reliable measure of audience approval because many movie averages are based on very small numbers of users.

A minimum rating-count requirement will therefore be considered before movies are classified or compared based on audience approval. The final threshold will be selected during later analysis instead of being imposed during Milestone 1.

## Step 9: Initial Cleaning Plan

The initial profiling showed several issues that need to be addressed before building the final analytical dataset.

The goal of the cleaning process is to make sure the data are consistent and usable while keeping as much useful information as possible.

### Keep One Row per Movie

The final dataset will have **one row per movie**.

The `ratings` table currently has multiple rows for the same movie because each row represents one user rating one movie. Before merging it with the movie metadata, the ratings will be aggregated by `movieId`.

Possible movie-level rating measures include:

- average user rating
- number of ratings
- rating standard deviation
- number of unique users
- first rating date
- last rating date

### Check the IDs Before Merging

The key audit showed no missing or duplicated values in the expected identifiers.

`movieId` will be used to connect ratings to movies, while `tmdbId` will be used to connect the credits and keywords tables.

The `links` table contains slightly more movie IDs than the other movie-level tables, so this difference will need to be checked before merging.

Row counts will also be checked after each merge to make sure no unexpected duplicate movies are created.

### Review Missing Values

Missing values will not automatically be removed.

Each variable will be reviewed based on how much data are missing, how important the variable is to the research question, and whether the missing values may have a reasonable explanation.

### Review Zero Budget and Revenue Values

Budget and revenue contain many zero values.

These zeros may mean that the information was unavailable or not reported, so they should not automatically be treated as real zero-dollar values.

They will need to be reviewed before these variables are used in the analysis.

### Investigate Runtime Values

Runtime also has some values that need more checking.

Some movies have a runtime of zero, and the maximum runtime is 1,140 minutes.

These values will be investigated before deciding whether they are valid or data-quality issues.

### Consider Release Year

The release dates were successfully converted, and the movies in the dataset range from 1902 to 2016.

This wide range may matter because movies released in different periods may have had different opportunities to receive ratings and attention.

Release year or release period may therefore be useful later in the analysis.

### Consider Rating Reliability

The number of ratings is very different across movies.

A movie with a high average rating based on a few users should not be treated the same as a movie with a similar average based on hundreds or thousands of ratings.

Because of this, a minimum rating-count threshold may be needed before defining strong audience approval.

The threshold will be selected after looking at the rating-count distribution.

### Decide How to Measure Audience Attention

The dataset has several possible measures of audience attention:

- `rating_count`
- `popularity`
- `vote_count`

The initial results show that these variables are related, but they are not measuring the same thing.

For that reason, the final measure of audience attention will be selected after looking more closely at their distributions and relationships.

### Avoid Information Leakage

If predictive modeling is used later, the selected features will depend on when the prediction is supposed to happen.

For example, if the goal is to predict audience response before release, variables such as revenue, popularity, vote count, and audience rating information should not be used because they become available after release.

All cleaning and transformation decisions will be documented so the final dataset can be reproduced from the raw files.

---

## Step 10: Potential Target and Analytical Direction

The main idea of the project is to compare **audience attention** with **audience approval**.

Audience approval will likely be measured using the average MovieLens user rating after the ratings are aggregated to the movie level.

Audience attention is less clear. `rating_count`, `popularity`, and `vote_count` are all possible measures, but the initial results show that they may represent different types of attention.

Because of this, I do not want to choose one final measure yet.

The project may later examine questions such as:

- How strongly are attention and approval related?
- Do movies with more attention also receive higher ratings?
- Are there movies with strong ratings but relatively low attention?
- Are there clear groups of movies with different attention and approval patterns?
- Which movie characteristics are associated with these patterns?

One group that may be interesting is movies that receive strong audience approval but limited attention. If this pattern appears clearly in the data, these movies could later be considered potential **hidden gems**.

The project may later use correlation analysis, group comparisons, regression, classification, or a combination of these methods depending on what the prepared data support.

No fixed threshold for **high attention**, **high approval**, or **hidden gem** will be selected at this stage. These definitions should be based on the data.

---

## Step 11: Ethics, Privacy, Governance, and Responsible Use

There are several issues that need to be considered when interpreting the results.

### User Privacy

The ratings table contains `userId` values and individual movie ratings.

These IDs will only be used as anonymous identifiers. There is no need to identify individual users or make assumptions about personal characteristics based on their ratings.

The final dataset will also be at the movie level, so individual user information will mainly be used to create aggregated movie-level measures.

### Representativeness

The users in this dataset do not necessarily represent all movie viewers.

Audience preferences can differ across countries, cultures, age groups, platforms, and time periods.

Because of this, the results should be interpreted as patterns found in this dataset and should not automatically be generalized to all movie audiences.

### Rating Reliability

Movies do not have the same number of ratings.

A high average rating based on a few users is less reliable than a similar average based on hundreds or thousands of users.

The number of ratings will therefore be considered when audience approval is interpreted.

### Exposure and Time

The dataset includes movies released over more than a century.

Movies released in different periods have had different amounts of time and exposure to receive ratings and attention.

Because of this, rating count or popularity may reflect more than audience interest alone.

### Association Does Not Mean Causation

The data are observational.

If relationships are found between genre, runtime, director, cast, keywords, popularity, and ratings, they will be treated as associations.

They will not be used to claim that one movie characteristic caused higher attention or stronger audience approval.

### What Does “Good Movie” Mean?

The title of the project asks:

**Is a Famous Movie Necessarily a Good Movie?**

However, the data cannot tell us whether a movie is objectively good.

In this project, “good” refers to **audience approval**, which will be measured using audience ratings.

The results therefore describe how audiences in this dataset responded to movies. They do not represent an objective measure of movie quality.

---

## Step 12: Milestone 1 Conclusion and Decisions Carried Forward

Milestone 1 helped define the main research question and showed what needs to be done before the final analysis can begin.

The project will examine the relationship between **audience attention** and **audience approval** without assuming that famous movies receive higher or lower ratings.

Several decisions can already be carried into the next stage:

- The final analytical dataset will have **one row per movie**.
- The ratings table must be aggregated before it is merged with the movie-level tables.
- `movieId` will connect ratings to movies.
- `tmdbId` will connect movies to credits and keywords.
- The key fields currently have no missing or duplicated values.
- Zero budget and revenue values need more investigation.
- Zero and unusually large runtime values also need to be checked.
- Average ratings need to be considered together with the number of ratings behind them.
- `rating_count`, `popularity`, and `vote_count` will be examined further before choosing the final measure of audience attention.
- Release period may need to be considered because movies have had different amounts of time and exposure.
- Relationships found in the data will be treated as associations and not causal effects.
- Definitions such as **high attention**, **high approval**, and **hidden gem** will be based on the data.

The next step will be to clean and combine the raw tables into one movie-level dataset. After that, the measures of audience attention and audience approval can be finalized and the main research question can be examined more closely.

In [23]:
# ============================================================
# MILESTONE 1 SUMMARY OF IDENTIFIED DATA ISSUES
# ============================================================

milestone1_issues = pd.DataFrame({
    "issue": [
        "Zero budget values",
        "Zero revenue values",
        "Zero runtime values",
        "Maximum runtime",
        "Duplicate movieId values",
        "Duplicate tmdbId values",
        "Duplicate userId-movieId rating keys"
    ],
    "value": [
        (pd.to_numeric(movies["budget"], errors="coerce") == 0).sum(),
        (pd.to_numeric(movies["revenue"], errors="coerce") == 0).sum(),
        (pd.to_numeric(movies["runtime"], errors="coerce") == 0).sum(),
        pd.to_numeric(movies["runtime"], errors="coerce").max(),
        movies["movieId"].duplicated().sum(),
        movies["tmdbId"].duplicated().sum(),
        ratings[["userId", "movieId"]].duplicated().sum()
    ]
})

display(milestone1_issues)

,issue,value
0,Zero budget values,4443.0
1,Zero revenue values,4322.0
2,Zero runtime values,20.0
3,Maximum runtime,1140.0
4,Duplicate movieId values,0.0
5,Duplicate tmdbId values,0.0
6,Duplicate userId-movieId rating keys,0.0


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [24]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [25]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [26]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


movie_rows_before_merges             9082
abt_rows_after_merges                9082
duplicate_movie_ids                     0
duplicate_tmdb_ids                      0
movies_with_rating_summary           9025
movies_eligible_for_rating_target    1300
dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,rating_target_eligible,high_user_rating
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,True,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,True,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,True,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,False,<NA>
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,True,0


,missing_rate
high_user_rating,0.856860
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
first_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276
primary_genre,0.003854
genres_list,0.003854


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [27]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check


,abt_rows,unique_movie_ids,target_not_defined_rows
0,9082,9082,7782


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [28]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,,,,
1,tmdbId,int64,,,,
2,imdb_id,str,,,,
3,title,str,,,,
4,original_title,str,,,,


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



## Initial Structural Profile

Before examining relationships between audience attention and audience approval, the structure of each raw dataset needs to be understood.

This first profile focuses on the basic shape of the data:

- number of rows
- number of columns
- data types
- fully duplicated records


In [29]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.
# ============================================================
# STEP 3: INITIAL STRUCTURAL PROFILE
# ============================================================

profile_rows = []

for name, df in tables.items():
    profile_rows.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum())
    })

structural_profile = pd.DataFrame(profile_rows)

display(structural_profile)

,table,rows,columns,duplicate_rows
0,movies,9082,21,0
1,ratings,99810,5,0
2,links,9125,3,0
3,credits,9082,3,0
4,keywords,9082,2,0


In [30]:
# Display data types for each table

for name, df in tables.items():
    print("\n" + "=" * 70)
    print(f"{name.upper()} DATA TYPES")
    print("=" * 70)

    display(
        df.dtypes
        .astype(str)
        .to_frame("data_type")
    )


MOVIES DATA TYPES


,data_type
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64



RATINGS DATA TYPES


,data_type
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]



LINKS DATA TYPES


,data_type
movieId,int64
imdbId,int64
tmdbId,float64



CREDITS DATA TYPES


,data_type
tmdbId,int64
director,str
top_cast,str



KEYWORDS DATA TYPES


,data_type
tmdbId,int64
keywords_list,str


In [31]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298937


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [32]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [33]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
